# Notebook 02 — Model Training

Train and compare all ASL detection models:
- **Custom CNN** (baseline)
- **MobileNetV2** (transfer learning)
- **EfficientNetB0** (transfer learning)
- **Skeleton GCN** (graph neural network)
- **Attention CNN** (with SE-Net style attention)

Early stopping, learning rate scheduling, and model checkpointing are applied throughout.


In [ ]:
import sys, os
sys.path.insert(0, '..')
os.makedirs('../results/models', exist_ok=True)
os.makedirs('../results/logs',   exist_ok=True)
os.makedirs('../results/plots',  exist_ok=True)

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.data.loader        import ASLDataLoader
from src.data.augmentation  import ASLAugmentation
from src.models.custom_cnn          import build_custom_cnn
from src.models.transfer_learning   import build_mobilenet, build_efficientnet
from src.models.skeleton_gcn        import build_skeleton_gcn
from src.models.attention           import build_attention_cnn
from src.training.trainer           import Trainer
from src.utils.config               import load_config

cfg = load_config('../config.yaml')
print('TensorFlow version:', tf.__version__)
print('GPUs available    :', tf.config.list_physical_devices('GPU'))

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 1. Load Data

In [ ]:
loader = ASLDataLoader(data_dir='../data/raw')
X_train, X_val, X_test, y_train, y_val, y_test = loader.load_and_split()
num_classes = len(loader.class_names)

aug = ASLAugmentation()
train_gen = aug.get_train_generator()
train_gen.fit(X_train)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Classes: {num_classes}')

## 2. Train Custom CNN (Baseline)

In [ ]:
cnn_model = build_custom_cnn(
    input_shape=(28, 28, 1),
    num_classes=num_classes,
    dropout_rate=cfg['model']['dropout_rate']
)
cnn_model.summary()

cnn_trainer = Trainer(cnn_model, config_path='../config.yaml', model_name='custom_cnn')
cnn_history = cnn_trainer.train(
    X_train, y_train, X_val, y_val,
    train_generator=train_gen
)
cnn_trainer.save_model('../results/models/custom_cnn_best.h5')
cnn_trainer.plot_history(save_path='../results/plots/custom_cnn_history.png')

## 3. Train MobileNetV2

In [ ]:
# MobileNetV2 needs 96x96 RGB images
from src.data.utils import resize_images_rgb
X_train_rgb = resize_images_rgb(X_train, size=96)
X_val_rgb   = resize_images_rgb(X_val,   size=96)

mobile_model = build_mobilenet(input_shape=(96, 96, 3), num_classes=num_classes)
mobile_trainer = Trainer(mobile_model, config_path='../config.yaml', model_name='mobilenet')
mobile_history = mobile_trainer.train(X_train_rgb, y_train, X_val_rgb, y_val)
mobile_trainer.save_model('../results/models/mobilenet_best.h5')
mobile_trainer.plot_history(save_path='../results/plots/mobilenet_history.png')

## 4. Train EfficientNetB0

In [ ]:
effnet_model = build_efficientnet(input_shape=(96, 96, 3), num_classes=num_classes)
effnet_trainer = Trainer(effnet_model, config_path='../config.yaml', model_name='efficientnet')
effnet_history = effnet_trainer.train(X_train_rgb, y_train, X_val_rgb, y_val)
effnet_trainer.save_model('../results/models/efficientnet_best.h5')
effnet_trainer.plot_history(save_path='../results/plots/efficientnet_history.png')

## 5. Train Attention CNN

In [ ]:
attn_model = build_attention_cnn(input_shape=(28, 28, 1), num_classes=num_classes)
attn_trainer = Trainer(attn_model, config_path='../config.yaml', model_name='attention_cnn')
attn_history = attn_trainer.train(
    X_train, y_train, X_val, y_val,
    train_generator=train_gen
)
attn_trainer.save_model('../results/models/attention_cnn_best.h5')
attn_trainer.plot_history(save_path='../results/plots/attention_cnn_history.png')

## 6. Train Skeleton GCN

In [ ]:
from src.models.skeleton_extraction import extract_skeleton_features

# Extract 21-keypoint features for GCN input
print('Extracting skeleton features (may take a few minutes)...')
X_skel_train = extract_skeleton_features(X_train)
X_skel_val   = extract_skeleton_features(X_val)
print(f'Skeleton feature shape: {X_skel_train.shape}')  # (N, 21, 2)

gcn_model = build_skeleton_gcn(num_nodes=21, num_features=2, num_classes=num_classes)
gcn_trainer = Trainer(gcn_model, config_path='../config.yaml', model_name='skeleton_gcn')
gcn_history = gcn_trainer.train(X_skel_train, y_train, X_skel_val, y_val)
gcn_trainer.save_model('../results/models/skeleton_gcn_best.h5')
gcn_trainer.plot_history(save_path='../results/plots/gcn_history.png')

## 7. Hyperparameter Tuning (Custom CNN)

In [ ]:
from src.training.hyperparameter_tuning import ASLHyperparameterTuner

tuner = ASLHyperparameterTuner(
    model_type='custom_cnn',
    input_shape=(28, 28, 1),
    num_classes=num_classes,
    max_trials=10,
    project_name='asl_cnn_tuning'
)
best_hps = tuner.search(X_train, y_train, X_val, y_val)
print('Best hyperparameters:', best_hps.values)

## Summary

All models have been trained and saved to `results/models/`. 
Training curves have been saved to `results/plots/`.
Proceed to **Notebook 03** for comparative evaluation.
